In [34]:
import pandas as pd

df = pd.read_csv(r"C:\Users\1\Desktop\项目\stock-data\Prices.csv")

print(df.head())
print(df.columns)
print(df.shape)

         Date       AAPL     AMZN      GOOGL        JNJ        JPM       META  \
0  2015-01-02  24.237549  15.4260  26.278944  76.955551  46.511124  77.905800   
1  2015-01-05  23.554735  15.1095  25.778227  76.418083  45.067196  76.654541   
2  2015-01-06  23.556952  14.7645  25.142035  76.042580  43.898651  75.621750   
3  2015-01-07  23.887281  14.9210  25.068092  77.721291  43.965630  75.621750   
4  2015-01-08  24.805077  15.0230  25.155436  78.332405  44.948116  77.637688   

        MSFT      NVDA         PG        XOM  
0  39.858452  0.483011  66.518349  57.916889  
1  39.491920  0.474853  66.202065  56.332203  
2  38.912289  0.460457  65.900513  56.032726  
3  39.406689  0.459257  66.246216  56.600468  
4  40.565952  0.476533  67.003761  57.542580  
Index(['Date', 'AAPL', 'AMZN', 'GOOGL', 'JNJ', 'JPM', 'META', 'MSFT', 'NVDA',
       'PG', 'XOM'],
      dtype='object')
(2765, 11)


In [1]:
# ==========================================
# Day3 升级版（推荐）
# 标签定义：未来20日最大回撤超过10% = crash
# ==========================================

import pandas as pd
import numpy as np

# ------------------------------------------
# 第1步：读取数据
# ------------------------------------------
df = pd.read_csv(
    r"C:\Users\1\Desktop\项目\stock-data\Prices.csv"
)

# 日期格式
df["Date"] = pd.to_datetime(df["Date"])

# ------------------------------------------
# 第2步：宽表 -> 长表
# ------------------------------------------
df_long = df.melt(
    id_vars="Date",
    var_name="symbol",
    value_name="close"
)

# 排序
df_long = df_long.sort_values(
    ["symbol", "Date"]
).reset_index(drop=True)

# ------------------------------------------
# 第3步：计算日收益率
# ------------------------------------------
df_long["return"] = df_long.groupby("symbol")["close"].pct_change()

# ------------------------------------------
# 第4步：未来20日最大回撤
# 定义：
# 从今天价格出发，未来20日最低价相对今天跌多少
# ------------------------------------------
def future_mdd(x):
    future_min = x.shift(-1).rolling(20).min()
    return future_min / x - 1

df_long["future_mdd_20"] = (
    df_long.groupby("symbol")["close"]
    .transform(future_mdd)
)

# ------------------------------------------
# 第5步：Crash标签
# 最大回撤超过10%
# ------------------------------------------
df_long["crash"] = np.where(
    df_long["future_mdd_20"] < -0.10,
    1,
    0
)

# ------------------------------------------
# 第6步：查看结果
# ------------------------------------------
print(df_long[[
    "Date",
    "symbol",
    "close",
    "return",
    "future_mdd_20",
    "crash"
]].head(20))

print("标签分布：")
print(df_long["crash"].value_counts())

# ------------------------------------------
# 第7步：各股票 crash 次数
# ------------------------------------------
print("各股票 crash 次数：")
print(
    df_long.groupby("symbol")["crash"]
    .sum()
    .sort_values(ascending=False)
)

# ------------------------------------------
# 第8步：保存文件
# ------------------------------------------
df_long.to_csv(
    r"C:\Users\1\Desktop\项目\stock-data\day3_dataset_mdd.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Day3 升级完成，已保存 day3_dataset_mdd.csv")

         Date symbol      close    return  future_mdd_20  crash
0  2015-01-02   AAPL  24.237549       NaN            NaN      0
1  2015-01-05   AAPL  23.554735 -0.028172            NaN      0
2  2015-01-06   AAPL  23.556952  0.000094            NaN      0
3  2015-01-07   AAPL  23.887281  0.014023            NaN      0
4  2015-01-08   AAPL  24.805077  0.038422            NaN      0
5  2015-01-09   AAPL  24.831692  0.001073            NaN      0
6  2015-01-12   AAPL  24.219816 -0.024641            NaN      0
7  2015-01-13   AAPL  24.434856  0.008879            NaN      0
8  2015-01-14   AAPL  24.341749 -0.003810            NaN      0
9  2015-01-15   AAPL  23.681108 -0.027140            NaN      0
10 2015-01-16   AAPL  23.497105 -0.007770            NaN      0
11 2015-01-20   AAPL  24.102312  0.025757            NaN      0
12 2015-01-21   AAPL  24.286325  0.007635            NaN      0
13 2015-01-22   AAPL  24.918142  0.026015            NaN      0
14 2015-01-23   AAPL  25.046726  0.00516